# 7.3 Transforming Higher Ed Text Data to Vectors


## Introduction


In the previous modules, we established the methodology and ethical frameworks required to handle student data responsibly. Now, we move into the technical core of analyzing student voice: **Text Vectorization**.

While Likert-scale questions provide structured data that is easy for machines to read, the most "human" part of a survey is the open-ended comment. However, computers cannot "read" text in its raw form; they require numbers. Vectorization is the process of transforming these qualitative student stories into a quantitative format that our Machine Learning models can process.

**In this module, we will explore:**
* The distinction between **Ordinal (Likert) data** and **Unstructured Text data**.
* The necessity of **Text Preprocessing** to reduce noise and standardize student responses.
* How to apply **CountVectorizer** to capture raw word frequencies.
* How to use **TF-IDF (Term Frequency-Inverse Document Frequency)** to identify the most distinctive and meaningful words in a student's response.
* The workflow for **merging text features** with structured demographic data to create a master feature matrix.

> **Instructor Perspective:** As an analyst, your goal is to preserve the "essence" of what the student said while converting it into a mathematical vector. Choosing the right vectorization method is the bridge between qualitative insight and predictive power.

**Guiding Questions**
1. How do we convert open-ended survey responses into numbers that machine learning models can use?
2. What do we gain (and lose) when we represent text as vectors?

**By the end of this notebook you will be able to:**
- Explain why raw text must be converted to numbers before modeling
- Recognize the difference between ordinal (Likert) and free-response items in survey data
- Apply `CountVectorizer` and `TfidfVectorizer` to student comment data
- Combine text vectors with structured survey features into a single analysis-ready DataFrame


## 1. Why Text Shows Up in Institutional Research

IR teams regularly work with instruments like the **NSSE** (National Survey of Student Engagement) and the **CIRP Freshman Survey**—both widely used tools for understanding student experience and engagement. These surveys collect a mix of **ordinal (Likert-style) items** and **free-response text**.


### Ordinal Data
An ordinal level of measurement is used when categories have a clear order or ranking, but the distance between adjacent categories is not known or consistent. For example, survey responses like “Strongly Disagree,” “Disagree,” “Agree,” and “Strongly Agree” follow a meaningful sequence, yet we cannot assume the gap between “Disagree” and “Agree” is the same as the gap between “Agree” and “Strongly Agree.” This differs from purely categorical data, where labels have no inherent order (such as race or major), and from numeric scales where equal spacing between values is assumed and arithmetic operations are meaningful (such as test scores or GPA). It also differs from measures that include a true zero point, where ratios are interpretable (for example, units completed or hours studied). Ordinal data sit in between: ordered, but not evenly spaced or fully quantitative.

### Free-Response Text Data
As you might expect, the structured items are easier to analyze since they're already numbers, though some nuance is needed since they are ordinal. The free-response comments are harder. Students write things like *"I feel overwhelmed by deadlines"* or *"My professors are incredibly supportive"*—rich qualitative information that can't be averaged or plotted directly.

This notebook shows how to bridge that gap by converting open-ended text into numeric vectors that downstream ML models can use.

## 2. Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [21]:
import pandas as pd
import numpy as np


pd.options.display.max_columns = None

## 3. Ordinal (Likert-Style) Survey Items

The dataset includes ten NSSE-like questions measured at the **ordinal** level:

1.  To what extent have you participated in collaborative learning activities?
2.  How often have you initiated discussion of course topics with faculty outside of class?
3.  How often have you worked with classmates on projects or assignments?
4.  To what extent has your coursework emphasized analytical and problem-solving skills?
5.  How often have you connected learning to societal problems or issues?
6.  How much has your institution emphasized providing support to help you succeed academically?
7.  To what extent have you gained a deeper understanding of people from different backgrounds?
8.  How often have you participated in co-curricular activities (e.g., clubs, organizations, events)?
9.  How often have you thought critically about your the requirements of your major?
10. To what extent has your academic experience contributed to your personal growth and development?


The questions are rated on a 1–4 scale:
**Survey Questions:**

| Value | Meaning |
|---|---|
| 1 | Never / Strongly Disagree |
| 2 | Sometimes / Disagree |
| 3 | Often / Agree |
| 4 | Very Often / Strongly Agree |

Because the responses are already stored as integers, they can be passed directly to scikit-learn models without any transformation. The table below maps each question to its NSSE category.


| Category | Survey Questions |
|----------|------------------|
| (1) Participation in educationally purposeful activities | Collaborative learning; working with classmates; connecting learning to societal problems |
| (2) Institutional requirements and coursework rigor | Analytical/problem-solving emphasis; critical thinking about major requirements |
| (3) Perceptions of the college environment | Academic support; co-curricular participation |
| (4) Educational and personal growth | Understanding diverse backgrounds; faculty interaction; personal development |
| (5) Background and demographic information | *(No questions from this category in this dataset)* |


## 4. Free-Response (Text) Survey Items

Beyond the Likert items, each student submitted one open-ended comment in the `Free_Response_Text` column. These comments capture nuance that the scaled questions can't—but they require several processing steps before they're usable in a model.

The code below loads the dataset and previews the full structure, including both the ordinal columns and the free-text column.


In [23]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
ML_Survey_Data = pd.read_csv(f'{filepath}ML_Survey_Data.csv')
ML_Survey_Data22 = pd.read_csv(f'{filepath}ML_Survey_Data22.csv')
display(ML_Survey_Data)

,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10,Free_Response_Text
0,UHDOP5522,3,3,3,3,2,2,3,2,3,3,"I feel prepared in some areas, but there are t..."
1,UHE842CU6,1,2,1,2,2,1,2,2,2,2,"I often start assignments too late, and then I..."
2,UHJFT1JAB,3,2,2,2,2,3,3,2,2,3,"When multiple deadlines hit at once, I have to..."
3,UHKF05TAF,3,3,3,3,2,3,3,2,3,3,"Group work can be helpful, but it depends on h..."
4,UHKKQ8UY5,3,3,3,2,2,3,2,2,3,2,Some classes feel more challenging than I expe...
...,...,...,...,...,...,...,...,...,...,...,...,...
19839,N2HOWBXJM,2,2,2,2,2,2,2,2,3,2,"The workload can feel heavy at times, but I ca..."
19840,N2JGRD0V6,4,3,3,3,3,3,3,3,3,3,Some classes feel more challenging than I expe...
19841,N2K6479P0,4,3,3,3,3,3,3,3,3,3,"College is an interesting experience, and I'm ..."
19842,N2LXOSYTW,1,1,1,1,1,1,1,1,1,1,Sometimes I don’t understand what instructors ...


## 5. Text Preprocessing

Because our ordinal columns are already coded 1–4, no further transformation is needed for those items. We treat them as numeric features and can include them directly in any model.

For the free-response text, we need to go further. Text data is unstructured: it has varying length, capitalization, punctuation, and filler words like *"the"* or *"and"* that carry little meaning for our models. **Preprocessing** standardizes the text before we vectorize it.

Key steps in text preprocessing:
- **Lowercasing** — treats "College" and "college" as the same word
- **Punctuation removal** — reduces noise from commas, periods, etc.
- **Tokenization** — splits text into individual words (tokens)
- **Stop word removal** — filters out common words like "the," "is," "a" that don't help differentiate responses
- **Stemming / Lemmatization** *(optional)* — reduces words to their root form (e.g., "running" → "run") to further reduce vocabulary size

Scikit-learn's `CountVectorizer` and `TfidfVectorizer` handle most of these steps automatically through their parameters.


## 6. Vectorization with CountVectorizer

**Count Vectorization** (also called Bag-of-Words) converts each student comment into a row of word counts. Every unique word in the corpus becomes a column; each cell holds the number of times that word appears in that response.

Pros:
- Simple and interpretable — word counts are easy to explain
- Fast and low-memory with sparse matrices

Cons:
- Ignores word order and context
- Treats all words equally, regardless of importance
- Produces high-dimensional sparse matrices


## 7. Vectorization with TF-IDF

**TF-IDF** (Term Frequency–Inverse Document Frequency) improves on raw counts by weighting each word by how distinctive it is across the corpus:

- **TF** *(term frequency)* — how often a word appears in *this* response
- **IDF** *(inverse document frequency)* — how rare the word is across *all* responses

A word like *"feedback"* that appears often in one response but rarely elsewhere gets a high TF-IDF score. A word like *"feel"* that appears in nearly every comment gets down-weighted automatically.

Pros:
- Better discriminatory power than raw counts
- Automatically reduces the influence of common words

Cons:
- Still ignores word order and semantic meaning
- Sensitive to corpus size — adding new documents changes IDF values


The code cells below apply both vectorizers to the `Free_Response_Text` column and display the resulting document-term matrices. Note that both produce the same shape (rows = responses, columns = vocabulary size), but with different values: `CountVectorizer` outputs raw counts while `TfidfVectorizer` outputs weighted scores.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Instantiate CountVectorizer with specified parameters
count_vectorizer = CountVectorizer(
    stop_words='english',
    lowercase=True,
    ngram_range=(1, 1)
)

# Apply CountVectorizer to the 'Free_Response_Text' column
count_matrix = count_vectorizer.fit_transform(ML_Survey_Data['Free_Response_Text'])

# Get feature names (vocabulary)
count_feature_names = count_vectorizer.get_feature_names_out()

# Convert the sparse matrix to a pandas DataFrame
df_count_vectorized = pd.DataFrame(count_matrix.toarray(), columns=count_feature_names)

# Display the first few rows and columns of the CountVectorized DataFrame
print("First few rows of CountVectorized DataFrame:")
df_count_vectorized.index = ML_Survey_Data.index
df_count_vectorized
print(f"\nShape of CountVectorized DataFrame: {df_count_vectorized.shape}")

Now we apply `TfidfVectorizer` with the same parameters. The shape of the output DataFrame will be identical to the count matrix, but each value is now a TF-IDF weight rather than a raw count.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Instantiate TfidfVectorizer with specified parameters
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    ngram_range=(1, 1)
)

# Apply TfidfVectorizer to the 'Free_Response_Text' column
tfidf_matrix = tfidf_vectorizer.fit_transform(ML_Survey_Data['Free_Response_Text'])

# Get feature names (vocabulary)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

# Convert the sparse matrix to a pandas DataFrame
df_tfidf_vectorized = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_feature_names)

# Display the first few rows and columns of the TfidfVectorized DataFrame
print("First few rows of TfidfVectorized DataFrame:")
df_tfidf_vectorized.index = ML_Survey_Data19.index
df_tfidf_vectorized
print(f"\nShape of TfidfVectorized DataFrame: {df_tfidf_vectorized.shape}")

## Helper Function for Text Vectorization

To streamline the process of vectorizing text data, we'll create a reusable function that applies both `CountVectorizer` and `TfidfVectorizer`. This function will take a DataFrame as input and return two new DataFrames: one with word counts and another with TF-IDF scores, based on the 'Free_Response_Text' column.

In [24]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def vectorize_text_data(dataframe):
    """
    Applies CountVectorizer and TfidfVectorizer to the 'Free_Response_Text' column
    of a given DataFrame.

    Args:
        dataframe (pd.DataFrame): The input DataFrame containing a 'Free_Response_Text' column.

    Returns:
        tuple: A tuple containing two pandas DataFrames:
               - df_count_vectorized: DataFrame with CountVectorizer results.
               - df_tfidf_vectorized: DataFrame with TfidfVectorizer results.
    """
    # Instantiate CountVectorizer with specified parameters
    count_vectorizer = CountVectorizer(
        stop_words='english',
        lowercase=True,
        ngram_range=(1, 1)
    )

    # Apply CountVectorizer
    count_matrix = count_vectorizer.fit_transform(dataframe['Free_Response_Text'])
    count_feature_names = count_vectorizer.get_feature_names_out()
    df_count_vectorized = pd.DataFrame(count_matrix.toarray(), columns=count_feature_names)
    df_count_vectorized.index = dataframe.index

    # Instantiate TfidfVectorizer with specified parameters
    tfidf_vectorizer = TfidfVectorizer(
        stop_words='english',
        lowercase=True,
        ngram_range=(1, 1)
    )

    # Apply TfidfVectorizer
    tfidf_matrix = tfidf_vectorizer.fit_transform(dataframe['Free_Response_Text'])
    tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
    df_tfidf_vectorized = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_feature_names)
    df_tfidf_vectorized.index = dataframe.index

    return df_count_vectorized, df_tfidf_vectorized

We'll first apply text vectorization to the training data:

In [26]:
df_count_vectorized, df_tfidf_vectorized = vectorize_text_data(ML_Survey_Data)
df_tfidf_vectorized

,ability,academic,academically,access,adapting,advance,advanced,affects,ahead,analysis,apply,appreciate,approaches,appropriately,areas,asking,aspects,assignment,assignments,attending,balancing,basics,best,better,build,busy,campus,carefully,catching,challenge,challenges,challenging,chosen,clarifying,class,classes,classmates,classroom,clear,clearer,click,coaching,college,comfortable,comments,compare,concepts,concrete,confidence,confident,connect,consistently,constraints,content,course,courses,coursework,day,deadlines,deeper,depends,determined,develop,discovering,discussions,doable,don,eager,earlier,early,easy,education,embarrassed,engaged,enjoy,especially,exams,excellence,excitement,exist,exists,expect,expectations,expected,experience,explore,fall,falling,family,far,fast,faster,feedback,feel,feeling,feels,felt,field,figure,figuring,finding,fine,finish,focus,footing,forward,foundational,fully,going,group,groups,grow,guidance,habits,hard,harder,heavy,help,helpful,helps,hesitate,higher,hit,hope,hours,improve,instead,instructor,instructors,interesting,journey,juggling,just,keeping,keeps,know,larger,late,learning,level,life,like,looking,lost,lot,make,makes,manage,managing,material,meet,mix,moments,motivated,moves,multiple,need,new,notes,office,organized,overall,overwhelmed,pace,participating,passionate,path,performance,personal,pile,plan,planning,positive,practice,prepared,prioritize,priority,problems,projects,pushes,questions,quickly,reach,refine,regularly,remember,require,resources,responsibilities,rest,results,review,reviewing,rewarding,rhythm,rigor,rigorous,routine,rubrics,rushing,scheduling,school,seeing,seeking,skills,soon,speak,specific,standard,start,stay,steps,strategies,stressed,strive,strong,structure,struggle,studies,study,studying,succeed,support,sure,survive,takes,taking,talking,team,tend,term,things,think,thinking,time,times,topics,tougher,track,transition,trying,tuning,turning,tutoring,understand,use,used,useful,usually,ve,want,way,week,weeks,wish,work,working,workload,writing
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.436342,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.251477,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.00000,0.323619,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.412782,0.388096,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.412782,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.387608,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.288040,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.405076,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.405076,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.405076,0.234447,0

Next we apply text vectorization to the test data:

In [ ]:
df_count_vectorized22, df_tfidf_vectorized22 = vectorize_text_data(ML_Survey_Data22)
df_count_vectorized22

## 8. Combining Text Vectors with Structured Student Data

With TF-IDF vectors in hand, we can merge them with the original NSSE ordinal scores to produce a single, fully numeric feature matrix — the `ML_Survey_Data19_Num` DataFrame. This merged DataFrame is what a downstream ML model (classifier, clustering algorithm, etc.) would consume.

The merge is straightforward: align on student ID (`SID`) and concatenate columns. The result keeps the 10 Likert-style features on the left and appends all TF-IDF word-weight columns on the right.


In [27]:
ML_Survey_Data.iloc[:, :11]

,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10
0,UHDOP5522,3,3,3,3,2,2,3,2,3,3
1,UHE842CU6,1,2,1,2,2,1,2,2,2,2
2,UHJFT1JAB,3,2,2,2,2,3,3,2,2,3
3,UHKF05TAF,3,3,3,3,2,3,3,2,3,3
4,UHKKQ8UY5,3,3,3,2,2,3,2,2,3,2
...,...,...,...,...,...,...,...,...,...,...,...
19839,N2HOWBXJM,2,2,2,2,2,2,2,2,3,2
19840,N2JGRD0V6,4,3,3,3,3,3,3,3,3,3
19841,N2K6479P0,4,3,3,3,3,3,3,3,3,3
19842,N2LXOSYTW,1,1,1,1,1,1,1,1,1,1


In [28]:
# Merge ML_Survey_Data19 and df_tfidf on their index values (SID)
df_merged_surv_features = pd.merge(ML_Survey_Data.iloc[:, :11], df_tfidf_vectorized, left_index=True, right_index=True)

# Display the first few rows of the merged DataFrame
print("First few rows of the merged DataFrame (Survey responses + tf-idf text features):")
display(df_merged_surv_features.head())

print(f"\nShape of the merged DataFrame: {df_merged_surv_features.shape}")

First few rows of the merged DataFrame (Survey responses + tf-idf text features):


,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10,ability,academic,academically,access,adapting,advance,advanced,affects,ahead,analysis,apply,appreciate,approaches,appropriately,areas,asking,aspects,assignment,assignments,attending,balancing,basics,best,better,build,busy,campus,carefully,catching,challenge,challenges,challenging,chosen,clarifying,class,classes,classmates,classroom,clear,clearer,click,coaching,college,comfortable,comments,compare,concepts,concrete,confidence,confident,connect,consistently,constraints,content,course,courses,coursework,day,deadlines,deeper,depends,determined,develop,discovering,discussions,doable,don,eager,earlier,early,easy,education,embarrassed,engaged,enjoy,especially,exams,excellence,excitement,exist,exists,expect,expectations,expected,experience,explore,fall,falling,family,far,fast,faster,feedback,feel,feeling,feels,felt,field,figure,figuring,finding,fine,finish,focus,footing,forward,foundational,fully,going,group,groups,grow,guidance,habits,hard,harder,heavy,help,helpful,helps,hesitate,higher,hit,hope,hours,improve,instead,instructor,instructors,interesting,journey,juggling,just,keeping,keeps,know,larger,late,learning,level,life,like,looking,lost,lot,make,makes,manage,managing,material,meet,mix,moments,motivated,moves,multiple,need,new,notes,office,organized,overall,overwhelmed,pace,participating,passionate,path,performance,personal,pile,plan,planning,positive,practice,prepared,prioritize,priority,problems,projects,pushes,questions,quickly,reach,refine,regularly,remember,require,resources,responsibilities,rest,results,review,reviewing,rewarding,rhythm,rigor,rigorous,routine,rubrics,rushing,scheduling,school,seeing,seeking,skills,soon,speak,specific,standard,start,stay,steps,strategies,stressed,strive,strong,structure,struggle,studies,study,studying,succeed,support,sure,survive,takes,taking,talking,team,tend,term,things,think,thinking,time,times,topics,tougher,track,transition,trying,tuning,turning,tutoring,understand,use,used,useful,usually,ve,want,way,week,weeks,wish,work,working,workload,writing
0,UHDOP5522,3,3,3,3,2,2,3,2,3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.436342,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.251477,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.00000,0.323619,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.412782,0.388096,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.412782,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.387608,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1,UHE842CU6,1,2,1,2,2,1,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.28804,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.405076,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.405076,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.405076,0.234447,0.0,0.0,


Shape of the merged DataFrame: (19844, 270)


In [29]:
# Merge ML_Survey_Data22 and df_tfidf on their index values (SID)
df_merged_surv_features22 = pd.merge(ML_Survey_Data22.iloc[:, :11], df_tfidf_vectorized22, left_index=True, right_index=True)

# Display the first few rows of the merged DataFrame
print("First few rows of the merged DataFrame (Survey responses + tf-idf text features):")
display(df_merged_surv_features22.head())

print(f"\nShape of the merged DataFrame: {df_merged_surv_features22.shape}")

First few rows of the merged DataFrame (Survey responses + tf-idf text features):


,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10,ability,academic,academically,access,adapting,advance,advanced,affects,ahead,analysis,apply,appreciate,approaches,appropriately,areas,asking,aspects,assignment,assignments,attending,balancing,basics,best,better,build,busy,campus,carefully,catching,challenge,challenges,challenging,chosen,clarifying,class,classes,classmates,classroom,clear,clearer,click,coaching,college,comfortable,comments,compare,concepts,concrete,confidence,confident,connect,consistently,constraints,content,course,courses,coursework,day,deadlines,deeper,depends,determined,develop,discovering,discussions,doable,don,eager,earlier,early,easy,education,embarrassed,engaged,enjoy,especially,exams,excellence,excitement,exist,exists,expect,expectations,expected,experience,explore,fall,falling,family,far,fast,faster,feedback,feel,feeling,feels,felt,field,figure,figuring,finding,fine,finish,focus,footing,forward,foundational,fully,going,group,groups,grow,guidance,habits,hard,harder,heavy,help,helpful,helps,hesitate,higher,hit,hope,hours,improve,instead,instructor,instructors,interesting,journey,juggling,just,keeping,keeps,know,larger,late,learning,level,life,like,looking,lost,lot,make,makes,manage,managing,material,meet,mix,moments,motivated,moves,multiple,need,new,notes,office,organized,overall,overwhelmed,pace,participating,passionate,path,performance,personal,pile,plan,planning,positive,practice,prepared,prioritize,priority,problems,projects,pushes,questions,quickly,reach,refine,regularly,remember,require,resources,responsibilities,rest,results,review,reviewing,rewarding,rhythm,rigor,rigorous,routine,rubrics,rushing,scheduling,school,seeing,seeking,skills,soon,speak,specific,standard,start,stay,steps,strategies,stressed,strive,strong,structure,struggle,studies,study,studying,succeed,support,sure,survive,takes,taking,talking,team,tend,term,things,think,thinking,time,times,topics,tougher,track,transition,trying,tuning,turning,tutoring,understand,use,used,useful,usually,ve,want,way,week,weeks,wish,work,working,workload,writing
0,UHNY4ZPLM,3,3,3,3,3,3,3,3,3,3,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.457644,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.4028,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.457644,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.457644,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.457644,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1,UI2LY561G,2,2,2,2,2,2,2,3,2,2,0.0,0.317202,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.394139,0.0,0.394139,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.300758,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Shape of the merged DataFrame: (5336, 270)


In [ ]:
import os

# Define the base path for your Google Drive
drive_path = '/content/drive/MyDrive/'

# Define the target directory for exporting the merged features
export_dir = os.path.join(drive_path, 'IR ML Cert/MLCert Course 3/Course 3 Data')

# Create the directory if it doesn't exist
os.makedirs(export_dir, exist_ok=True)

# Define the full file path for the CSV
output_filename = 'ML_Survey_Data_Num.csv'
output_filepath = os.path.join(export_dir, output_filename)

# Export the DataFrame to CSV without the index (as SID is a column)
df_merged_surv_features.to_csv(output_filepath, index=False)

print(f"DataFrame successfully exported to: {output_filepath}")

In [ ]:
# Define the full file path for the CSV
output_filename = 'ML_Survey_Data22_Num.csv'
output_filepath = os.path.join(export_dir, output_filename)

# Export the DataFrame to CSV without the index (as SID is a column)
df_merged_surv_features22.to_csv(output_filepath, index=False)

print(f"DataFrame successfully exported to: {output_filepath}")

## 9. Wrap-Up

**What you did:**
- Reviewed the two main item types in NSSE-style surveys: ordinal items (already numeric) and free-response text (requires transformation)
- Applied `CountVectorizer` and `TfidfVectorizer` to convert student comments into document-term matrices
- Merged text vectors with structured survey features to create a single analysis-ready DataFrame

**Key decision points when working with survey text:**
- **CountVectorizer vs TF-IDF:** Start with TF-IDF — it automatically down-weights common words and typically performs better on downstream tasks. Use Count vectors when raw frequency matters (e.g., as input to LDA topic modeling).
- **Stop words and n-grams:** Always remove stop words for survey data. Consider `ngram_range=(1,2)` to capture common two-word phrases like "office hours" or "study group."
- **What you lose:** Both methods ignore word order and semantic meaning. Phrases like "not helpful" and "very helpful" will look similar to a bag-of-words model.

**Coming up:** 7.4 uses these same vectors as input to topic modeling — automatically discovering the main themes in student comments without reading every response.
